In [1]:
!pip install requests beautifulsoup4 tqdm

In [2]:
import json

with open("myscheme_schemes.json") as f:
    schemes = json.load(f)

print("Total schemes:", len(schemes))
print(schemes[0])

Total schemes: 4645
{'name': 'National Overseas Scholarship For Students With Disabilities', 'slug': 'nos-swd', 'description': "A scholarship scheme by the Ministry of Social Justice & Empowerment for regular, full-time students with disabilities to obtain higher education viz., Master's degree, or Ph.D. courses from foreign universities, in one of the specified fields of study.", 'ministry': 'Ministry Of Social Justice and Empowerment'}


In [3]:
for s in schemes:
    s["url"] = f"https://www.myscheme.gov.in/schemes/{s['slug']}"

In [4]:
schemes[0]["url"]

'https://www.myscheme.gov.in/schemes/nos-swd'

In [5]:
import requests
from bs4 import BeautifulSoup

def scrape_scheme(url):

    headers = {
        "User-Agent": "Mozilla/5.0"
    }

    html = requests.get(url, headers=headers).text

    soup = BeautifulSoup(html, "html.parser")

    text = soup.get_text(separator="\n")

    return text

In [6]:
text = scrape_scheme(schemes[0]["url"])

print(text[:2000])

Something went wrong. Please try again later.
Ok
Are you sure you want to sign out?
Cancel
Sign Out
|
|
English
English
Theme
Sign In
©
2026
Powered by
 Digital India Corporation(DIC)
Ministry of Electronics & IT (MeitY)
Government of India
®
Connect on Social Media
Quick Links
About Us
Contact Us
Screen Reader
Accessibility Statement
Frequently Asked Questions
Disclaimer
Terms & Conditions
Dashboard
Useful Links
Get in touch
4th Floor, NeGD, Electronics Niketan, 6 CGO Complex, Lodhi Road, New Delhi - 110003, India
support-myscheme[at]digitalindia[dot]gov[dot]in
(011) 24303714 (9:00 AM to 5:30 PM)
Last Updated On
 : 
06/03/2026
 | v-
3.1.6


In [7]:
def extract_sections(text):

    sections = {}

    keywords = [
        "Benefits",
        "Eligibility",
        "Application Process",
        "Documents Required"
    ]

    for k in keywords:

        if k in text:
            sections[k] = text.split(k)[1][:700]

    return sections

In [8]:
extract_sections(text)

{}

In [9]:
from tqdm import tqdm

dataset = []

for s in tqdm(schemes[:50]):

    try:

        text = scrape_scheme(s["url"])

        sections = extract_sections(text)

        dataset.append({

            "name": s["name"],
            "slug": s["slug"],
            "description": s["description"],
            "benefits": sections.get("Benefits"),
            "eligibility": sections.get("Eligibility"),
            "application_process": sections.get("Application Process"),
            "documents": sections.get("Documents Required"),
            "url": s["url"]

        })

    except:
        pass

100%|██████████████████████████████████████████████████████████████████████████████████| 50/50 [00:17<00:00,  2.83it/s]


In [10]:
import json

with open("schemes_full.json","w") as f:
    json.dump(dataset,f,indent=2)

In [11]:
import json

with open("schemes_full.json","w") as f:
    json.dump(dataset,f,indent=2)

In [ ]:
import requests
from bs4 import BeautifulSoup

def scrape_scheme(url):

    headers = {"User-Agent": "Mozilla/5.0"}

    r = requests.get(url, headers=headers)
    soup = BeautifulSoup(r.text, "html.parser")

    data = {
        "benefits": None,
        "eligibility": None,
        "application_process": None,
        "documents": None
    }

    sections = soup.find_all(["h2","h3"])

    for sec in sections:

        title = sec.text.strip().lower()

        content = ""

        for sib in sec.find_next_siblings():
            if sib.name in ["h2","h3"]:
                break
            content += sib.get_text(" ",strip=True) + " "

        if "benefit" in title:
            data["benefits"] = content

        elif "eligibility" in title:
            data["eligibility"] = content

        elif "application process" in title:
            data["application_process"] = contentimport requests
import re

html = requests.get("https://www.myscheme.gov.in").text

build_id = re.search(r'"buildId":"(.*?)"', html).group(1)

print(build_id)

        elif "document" in title:
            data["documents"] = content

    return data

In [13]:
import requests
import re

session = requests.Session()

headers = {
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/json",
    "Referer": "https://www.myscheme.gov.in/",
    "x-nextjs-data": "1"
}

# Step 1: Load homepage
html = session.get("https://www.myscheme.gov.in", headers=headers).text

# Step 2: Extract buildId
build_id = re.search(r'"buildId":"(.*?)"', html).group(1)

print("Build ID:", build_id)

# Step 3: Request Next.js JSON
url = f"https://www.myscheme.gov.in/_next/data/{build_id}/index.json"

response = session.get(url, headers=headers)

print(response.status_code)
print(response.text[:500])

Build ID: 2An3n2LksAa0RvBMkaHhs
200
{}


In [14]:
import requests

url = "https://api.myscheme.gov.in/search/v4/schemes"

params = {
    "page": 1,
    "size": 10
}

response = requests.get(url, params=params)

print(response.status_code)

data = response.json()

for scheme in data["data"]:
    print(scheme["schemeName"])

401


KeyError: 'data'

In [10]:
headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers)

print(response.status_code)
print(response.text[:200])

404
<!DOCTYPE html><html lang="en"><head><meta charSet="utf-8"/><meta name="viewport" content="width=device-width"/><link rel="preload" as="image" imageSrcSet="https://cdn.myscheme.in/images/logos/emblem-


In [9]:
import requests

slug = "pmsby"
build_id = "YOUR_BUILD_ID"

url = f"https://www.myscheme.gov.in/_next/data/{build_id}/en/schemes/{slug}.json"

data = requests.get(url).json()

print(data["pageProps"]["scheme"])

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [8]:
scheme = data["pageProps"]["scheme"]

record = {
    "name": scheme.get("schemeName"),
    "description": scheme.get("briefDescription"),
    "benefits": scheme.get("benefits"),
    "eligibility": scheme.get("eligibility"),
    "documents": scheme.get("documentsRequired"),
    "application_process": scheme.get("applicationProcess"),
}

NameError: name 'data' is not defined

In [7]:
from tqdm import tqdm
import json

dataset = []

for s in tqdm(schemes):

    slug = s["slug"]

    try:

        url = f"https://www.myscheme.gov.in/_next/data/{build_id}/en/schemes/{slug}.json"

        data = requests.get(url).json()

        scheme = data["pageProps"]["scheme"]

        dataset.append({
            "name": scheme.get("schemeName"),
            "slug": slug,
            "description": scheme.get("briefDescription"),
            "benefits": scheme.get("benefits"),
            "eligibility": scheme.get("eligibility"),
            "documents": scheme.get("documentsRequired"),
            "application_process": scheme.get("applicationProcess"),
            "url": f"https://www.myscheme.gov.in/schemes/{slug}"
        })

    except:
        pass

NameError: name 'schemes' is not defined